# ARC/ATLAS v4 nnU-Net follow-up experiments

This notebook keeps the current v4 dataset and sets up the next nnU-Net experiments after the first completed 5-fold run.

The immediate goal is not to move to a different `data/nnunet` dataset. Everything here uses the existing v4 split under:

`/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random`


## Fresh comparison notes

Current v4 nnU-Net run:

- Dataset: mixed ARC + ATLAS + approximate cases from `90_10_random/train`.
- Model: standard nnU-Net v2 `nnUNetTrainer`, 3D full-resolution `PlainConvUNet`.
- Plan: `nnUNetPlans_24GB`.
- Patch size: `[160, 192, 160]`; batch size: `3`.
- Planned median cropped size: `[156, 186, 148]`; spacing `[1, 1, 1]`.
- Five validation folds ended around mean Dice `0.5233` across folds.
- Raw v4 held-out evaluation was about mean Dice `0.4171`, median `0.5279`; ATLAS subset was about mean `0.3669`, median `0.3630`.
- The custom probability postprocessing path was harmful for this run and should stay off until fixed.

v3 HiRes run:

- Model: custom Keras model from `ARC_ATLAS_Train_v3/runs/20260410_174944`.
- Input grid: fixed `[192, 224, 192, 1]`.
- Training set: `train_hires`, 522 cases; held-out set: `test_hires`, 138 cases.
- Loss/training differed materially: Dice + boundary loss, dropout/L2, augmentation intensity, synthetic lesion probability, and the custom architecture rather than a stock nnU-Net.
- v3 reported about mean hard Dice `0.5623`, median about `0.6459` on `test_hires`; ATLAS-only mean was about `0.5002`, median about `0.5485`.

The current completed v4 run overlaps the v3 `test_hires` subjects, so testing it there is a sanity check, not a clean held-out comparison. The clean apples-to-apples experiment is Experiment 0 below: rebuild the v4 nnU-Net dataset after excluding all v3 `test_hires` subjects, train, then evaluate externally on `ARC_ATLAS_Test_v4.ipynb`.


## Patching vs cropping

nnU-Net does both, but they mean different things:

- Cropping: nnU-Net crops the nonzero foreground region during preprocessing/inference to remove empty background. That is where the planned median cropped size `[156, 186, 148]` comes from.
- Patching: during training it samples 3D patches from the preprocessed/cropped case. During inference it uses sliding-window patches over the whole preprocessed/cropped region, blends the overlapping outputs, and writes the prediction back to the original image geometry.

So patching does not mean only one crop is predicted and the rest of the brain is ignored. The full foreground region is tiled at inference. Areas outside the foreground crop are restored as background in the exported full-size mask.

One caveat: if brain voxels are exact zero in the source image, nnU-Net's nonzero crop can treat them as background. For these MNI-normalized T1 files, that is usually intended skull/background behavior, but it is worth checking if a case has zero-valued brain tissue.


In [2]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

import nibabel as nib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path('/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4')
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

EXPECTED_ENV_HINT = '/miniconda3/envs/tf_310/'
if EXPECTED_ENV_HINT not in sys.executable:
    raise RuntimeError(
        f'Wrong Python environment: {sys.executable}\n'
        f'Use the tf_310 conda kernel, not the PyCharm environment.'
    )

import atlas_nnunet_pipeline as pipe

print('Python:', sys.executable)
print('Project:', PROJECT_ROOT)


Python: /home/rbielski/miniconda3/envs/tf_310/bin/python
Project: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4


In [3]:
TRAIN_MANIFEST = PROJECT_ROOT / 'data/splits/90_10_random/train/manifest.csv'
V4_HELDOUT_MANIFEST = PROJECT_ROOT / 'data/splits/90_10_random/test/manifest.csv'

NNUNET_ROOT = PROJECT_ROOT / 'data/splits/90_10_random/train/nnunet_view'
layout = pipe.NnUNetLayout(
    project_root=PROJECT_ROOT,
    nnunet_root=NNUNET_ROOT,
    dataset_id=701,
    dataset_name='ARC_ATLAS_TrainV4Native',
)

CONFIGURATION = '3d_fullres'
BASE_PLANS_NAME = 'nnUNetPlans_24GB'
FULLGRID_PLANS_NAME = 'nnUNetPlans_fullgrid192_bs1'
FULLGRID_PATCH = [192, 224, 192]
FULLGRID_BATCH_SIZE = 1
TRAIN_FOLDS = (0,)

# Long-running controls. Leave False until you intentionally launch that experiment.
RUN_FULLGRID_TRAINING = False
RUN_OVERSAMPLING_TRAINING = False
RUN_ATLAS_ONLY_PREP = True
RUN_ATLAS_ONLY_PLAN = True
RUN_ATLAS_ONLY_TRAINING = True

print('Train manifest:', TRAIN_MANIFEST)
print('Current nnU-Net root:', NNUNET_ROOT)


Train manifest: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/manifest.csv
Current nnU-Net root: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/nnunet_view


In [4]:
OVERSAMPLING_TRAINER = 'nnUNetTrainer_probabilisticOversampling_033'

def run_nnunet_train(layout: pipe.NnUNetLayout, fold: int, plans_name: str, trainer: str | None = None, save_npz: bool = True, dry_run: bool = False):
    cmd = ['nnUNetv2_train', str(layout.dataset_id), CONFIGURATION, str(fold), '-p', plans_name]
    if trainer:
        cmd.extend(['-tr', trainer])
    if save_npz:
        cmd.append('--npz')
    return pipe.run_command(cmd, layout, dry_run=dry_run)


In [5]:
# Dataset and plan summary.
for label, path in [('train', TRAIN_MANIFEST), ('v4_heldout', V4_HELDOUT_MANIFEST)]:
    df = pd.read_csv(path)
    print(f'\n{label}: n={len(df)}')
    print(df['slug'].value_counts().to_string())

plans_path = layout.preprocessed_dataset_dir / f'{BASE_PLANS_NAME}.json'
plans = json.loads(plans_path.read_text())
conf = plans['configurations'][CONFIGURATION]
print('\nCurrent plan:', plans_path)
print('patch_size:', conf['patch_size'])
print('batch_size:', conf['batch_size'])
print('median_image_size_in_voxels:', conf['median_image_size_in_voxels'])
print('spacing:', conf['spacing'])
print('data_identifier:', conf['data_identifier'])
print('network:', conf['architecture']['network_class_name'])
print('features:', conf['architecture']['arch_kwargs']['features_per_stage'])



train: n=866
slug
ATLAS-Images-f0d7431e           582
ARC-combined-t1-raw-ab0d1794    190
Approx-Numeracy-Processed        94

v4_heldout: n=96
slug
ATLAS-Images-f0d7431e           73
ARC-combined-t1-raw-ab0d1794    13
Approx-Numeracy-Processed       10

Current plan: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/nnunet_view/nnUNet_preprocessed/Dataset701_ARC_ATLAS_TrainV4Native/nnUNetPlans_24GB.json
patch_size: [160, 192, 160]
batch_size: 3
median_image_size_in_voxels: [156.0, 186.0, 148.0]
spacing: [1.0, 1.0, 1.0]
data_identifier: nnUNetPlans_24GB_3d_fullres
network: dynamic_network_architectures.architectures.unet.PlainConvUNet
features: [32, 64, 128, 256, 320, 320]


In [6]:
# Validate raw image shapes and the planned cropped size.
train_map = pd.read_csv(layout.mapping_csv)
shape_counts = train_map['shape'].value_counts()
spacing_counts = train_map['spacing'].value_counts()
print('Raw/source shape counts from case_mapping.csv:')
print(shape_counts.head(10).to_string())
print('\nSpacing counts:')
print(spacing_counts.head(10).to_string())
print('\nConfirmed planned median cropped size:', conf['median_image_size_in_voxels'])


Raw/source shape counts from case_mapping.csv:
shape
193x229x193    866

Spacing counts:
spacing
1x1x1    866

Confirmed planned median cropped size: [156.0, 186.0, 148.0]


In [7]:
# VRAM estimate from the completed run.
old_patch = np.array(conf['patch_size'], dtype=np.int64)
old_batch = int(conf['batch_size'])
old_batch_voxels = int(np.prod(old_patch) * old_batch)
full_patch = np.array(FULLGRID_PATCH, dtype=np.int64)
full_batch1_voxels = int(np.prod(full_patch) * 1)
full_batch2_voxels = int(np.prod(full_patch) * 2)

# Last observed training was about 21.7 GiB on a 24 GiB RTX 4090 with [160,192,160] batch 3.
observed_gib = 21.7
est_batch1 = observed_gib * full_batch1_voxels / old_batch_voxels
est_batch2 = observed_gib * full_batch2_voxels / old_batch_voxels

print('Old patch x batch voxels:', old_batch_voxels)
print('Fullgrid batch=1 voxels:', full_batch1_voxels, f'({full_batch1_voxels / old_batch_voxels:.2%} of old load)')
print('Fullgrid batch=2 voxels:', full_batch2_voxels, f'({full_batch2_voxels / old_batch_voxels:.2%} of old load)')
print(f'Back-of-envelope VRAM, batch=1: {est_batch1:.1f} GiB')
print(f'Back-of-envelope VRAM, batch=2: {est_batch2:.1f} GiB')
print('\nInterpretation: [192,224,192] batch=1 should probably fit. batch=2 is close enough to be risky. Start with batch=1.')


Old patch x batch voxels: 14745600
Fullgrid batch=1 voxels: 8257536 (56.00% of old load)
Fullgrid batch=2 voxels: 16515072 (112.00% of old load)
Back-of-envelope VRAM, batch=1: 12.2 GiB
Back-of-envelope VRAM, batch=2: 24.3 GiB

Interpretation: [192,224,192] batch=1 should probably fit. batch=2 is close enough to be risky. Start with batch=1.


In [8]:
def write_patch_plan(layout: pipe.NnUNetLayout, base_plans_name: str, new_plans_name: str, patch_size: list[int], batch_size: int) -> Path:
    base_path = layout.preprocessed_dataset_dir / f'{base_plans_name}.json'
    out_path = layout.preprocessed_dataset_dir / f'{new_plans_name}.json'
    plan = json.loads(base_path.read_text())
    plan['plans_name'] = new_plans_name
    cfg = plan['configurations'][CONFIGURATION]
    cfg['patch_size'] = [int(x) for x in patch_size]
    cfg['batch_size'] = int(batch_size)

    # Keep data_identifier pointing at the already-preprocessed arrays. Patch size changes training sampling only;
    # spacing/normalization/preprocessor stay the same.
    cfg['data_identifier'] = plans['configurations'][CONFIGURATION]['data_identifier']
    out_path.write_text(json.dumps(plan, indent=2))
    return out_path

fullgrid_plan_path = write_patch_plan(layout, BASE_PLANS_NAME, FULLGRID_PLANS_NAME, FULLGRID_PATCH, FULLGRID_BATCH_SIZE)
print('Wrote full-grid-ish patch plan:', fullgrid_plan_path)
print(json.dumps(json.loads(fullgrid_plan_path.read_text())['configurations'][CONFIGURATION], indent=2)[:2500])


Wrote full-grid-ish patch plan: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/nnunet_view/nnUNet_preprocessed/Dataset701_ARC_ATLAS_TrainV4Native/nnUNetPlans_fullgrid192_bs1.json
{
  "data_identifier": "nnUNetPlans_24GB_3d_fullres",
  "preprocessor_name": "DefaultPreprocessor",
  "batch_size": 1,
  "patch_size": [
    192,
    224,
    192
  ],
  "median_image_size_in_voxels": [
    156.0,
    186.0,
    148.0
  ],
  "spacing": [
    1.0,
    1.0,
    1.0
  ],
  "normalization_schemes": [
    "ZScoreNormalization"
  ],
  "use_mask_for_norm": [
    true
  ],
  "resampling_fn_data": "resample_data_or_seg_to_shape",
  "resampling_fn_seg": "resample_data_or_seg_to_shape",
  "resampling_fn_data_kwargs": {
    "is_seg": false,
    "order": 3,
    "order_z": 0,
    "force_separate_z": null
  },
  "resampling_fn_seg_kwargs": {
    "is_seg": true,
    "order": 1,
    "order_z": 0,
    "force_separate_z": null
  },
  "resampling_fn_probabilitie

## Experiment 0: clean v3-heldout retrain

The current completed v4 model was trained on a random 90/10 split, not on the v3 `test_hires` split. A subject-overlap check shows most v3 test subjects are already present in the current v4 training pool, so evaluating that model on v3 `test_hires` is useful for a sanity check but not for an honest held-out estimate.

This experiment builds a new nnU-Net dataset from the same v4 source manifests after excluding every subject in v3 `test_hires`. Use this when you want the clean apples-to-apples answer: train on all non-v3-heldout v4 rows, then evaluate externally on the v3 `test_hires` notebook.


In [9]:
import re

V3_HIRES_MANIFEST = Path('/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_HiresLowres_Split_Data/test_hires.csv')
CLEAN_SPLIT_DIR = PROJECT_ROOT / 'data/splits/v3_hires_heldout_clean'
CLEAN_TRAIN_MANIFEST = CLEAN_SPLIT_DIR / 'train/manifest.csv'
CLEAN_EXTERNAL_MANIFEST = CLEAN_SPLIT_DIR / 'external_test/v3_test_hires_manifest_for_nnunet.csv'
clean_layout = pipe.NnUNetLayout(
    project_root=PROJECT_ROOT,
    nnunet_root=CLEAN_SPLIT_DIR / 'train/nnunet_view',
    dataset_id=703,
    dataset_name='ARC_ATLAS_TrainV4Native_NoV3HiResHeldout',
)
CLEAN_PLANS_NAME = 'nnUNetPlans_24GB'
CLEAN_FULLGRID_PLANS_NAME = 'nnUNetPlans_fullgrid192_bs1'

RUN_BUILD_CLEAN_MANIFESTS = False
RUN_CLEAN_PREP = False
RUN_CLEAN_PLAN = False
RUN_CLEAN_FULLGRID_TRAINING = False
RUN_CLEAN_OVERSAMPLING_TRAINING = False


def subject_id(value: str) -> str:
    m = re.search(r'(sub-[A-Za-z0-9]+(?:_ses-[A-Za-z0-9]+)?)', str(value))
    return m.group(1) if m else str(value)

if RUN_BUILD_CLEAN_MANIFESTS:
    v3 = pd.read_csv(V3_HIRES_MANIFEST)
    v3_subjects = set(v3['key'].map(subject_id))
    pool = pd.concat([
        pd.read_csv(TRAIN_MANIFEST).assign(source_split='v4_90_10_train'),
        pd.read_csv(V4_HELDOUT_MANIFEST).assign(source_split='v4_90_10_test'),
    ], ignore_index=True)
    pool['subject_id'] = pool['key'].map(subject_id)
    leaked = pool[pool['subject_id'].isin(v3_subjects)].copy()
    clean = pool[~pool['subject_id'].isin(v3_subjects)].copy()

    CLEAN_TRAIN_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    CLEAN_EXTERNAL_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    clean[['slug', 'key', 't1', 'mask']].to_csv(CLEAN_TRAIN_MANIFEST, index=False)
    pd.DataFrame({
        'slug': v3['dataset'].astype(str).str.upper().map({'ARC': 'ARC-v3-test_hires', 'ATLAS': 'ATLAS-v3-test_hires'}).fillna(v3['dataset'].astype(str) + '-v3-test_hires'),
        'key': v3['key'].astype(str),
        't1': v3['t1_path'].astype(str),
        'mask': v3['mask_path'].astype(str),
    }).to_csv(CLEAN_EXTERNAL_MANIFEST, index=False)

    print('v4 source pool rows:', len(pool))
    print('excluded rows overlapping v3 test_hires subjects:', len(leaked))
    print('clean training rows:', len(clean))
    print('\nClean training sources:')
    print(clean['slug'].value_counts().to_string())
    print('\nExcluded overlap sources:')
    print(leaked['slug'].value_counts().to_string())
    print('\nWrote:', CLEAN_TRAIN_MANIFEST)
    print('Wrote:', CLEAN_EXTERNAL_MANIFEST)

if RUN_CLEAN_PREP:
    summary = pipe.prepare_nnunet_dataset(
        CLEAN_TRAIN_MANIFEST,
        clean_layout,
        overwrite=True,
        n_splits=5,
    )
    print(json.dumps(summary, indent=2))
else:
    print('RUN_CLEAN_PREP=False. Clean layout:', clean_layout.nnunet_root)

if RUN_CLEAN_PLAN:
    pipe.plan_and_preprocess(
        clean_layout,
        verify=True,
        configurations=(CONFIGURATION,),
        gpu_memory_target=24,
        overwrite_plans_name=CLEAN_PLANS_NAME,
    )
else:
    print('RUN_CLEAN_PLAN=False.')


RUN_CLEAN_PREP=False. Clean layout: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/v3_hires_heldout_clean/train/nnunet_view
RUN_CLEAN_PLAN=False.


In [10]:
def write_clean_fullgrid_plan():
    base = clean_layout.preprocessed_dataset_dir / f'{CLEAN_PLANS_NAME}.json'
    if not base.exists():
        print('Clean base plan does not exist yet:', base)
        return None
    plan = json.loads(base.read_text())
    original_data_identifier = plan['configurations'][CONFIGURATION]['data_identifier']
    plan['plans_name'] = CLEAN_FULLGRID_PLANS_NAME
    cfg = plan['configurations'][CONFIGURATION]
    cfg['patch_size'] = FULLGRID_PATCH
    cfg['batch_size'] = FULLGRID_BATCH_SIZE
    cfg['data_identifier'] = original_data_identifier
    out = clean_layout.preprocessed_dataset_dir / f'{CLEAN_FULLGRID_PLANS_NAME}.json'
    out.write_text(json.dumps(plan, indent=2))
    print('Wrote:', out)
    return out

clean_fullgrid_plan = write_clean_fullgrid_plan()

if RUN_CLEAN_FULLGRID_TRAINING:
    for fold in TRAIN_FOLDS:
        run_nnunet_train(clean_layout, fold, CLEAN_FULLGRID_PLANS_NAME, trainer=None, save_npz=True)
else:
    print('RUN_CLEAN_FULLGRID_TRAINING=False. Commands to run after prep/plan:')
    for fold in TRAIN_FOLDS:
        print(f'nnUNetv2_train {clean_layout.dataset_id} {CONFIGURATION} {fold} -p {CLEAN_FULLGRID_PLANS_NAME} --npz')

if RUN_CLEAN_OVERSAMPLING_TRAINING:
    for fold in TRAIN_FOLDS:
        run_nnunet_train(clean_layout, fold, CLEAN_FULLGRID_PLANS_NAME, trainer=OVERSAMPLING_TRAINER, save_npz=True)
else:
    print('\nRUN_CLEAN_OVERSAMPLING_TRAINING=False. Commands to run after prep/plan:')
    for fold in TRAIN_FOLDS:
        print(f'nnUNetv2_train {clean_layout.dataset_id} {CONFIGURATION} {fold} -p {CLEAN_FULLGRID_PLANS_NAME} -tr {OVERSAMPLING_TRAINER} --npz')


Clean base plan does not exist yet: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/v3_hires_heldout_clean/train/nnunet_view/nnUNet_preprocessed/Dataset703_ARC_ATLAS_TrainV4Native_NoV3HiResHeldout/nnUNetPlans_24GB.json
RUN_CLEAN_FULLGRID_TRAINING=False. Commands to run after prep/plan:
nnUNetv2_train 703 3d_fullres 0 -p nnUNetPlans_fullgrid192_bs1 --npz

RUN_CLEAN_OVERSAMPLING_TRAINING=False. Commands to run after prep/plan:
nnUNetv2_train 703 3d_fullres 0 -p nnUNetPlans_fullgrid192_bs1 -tr nnUNetTrainer_probabilisticOversampling_033 --npz


## Experiment 1: full v3-style patch grid

This keeps the same v4 dataset and the same preprocessing, but changes training patch size from `[160, 192, 160]` to `[192, 224, 192]` with batch size `1`.

This does not disable nnU-Net's foreground crop. It makes each training patch match the v3 fixed grid, and at inference nnU-Net still tiles/blends the whole foreground crop and restores the full image shape.


In [11]:
if RUN_FULLGRID_TRAINING:
    for fold in TRAIN_FOLDS:
        run_nnunet_train(layout, fold, FULLGRID_PLANS_NAME, trainer=None, save_npz=True)
else:
    print('RUN_FULLGRID_TRAINING=False. Commands to run:')
    for fold in TRAIN_FOLDS:
        print(f'nnUNetv2_train {layout.dataset_id} {CONFIGURATION} {fold} -p {FULLGRID_PLANS_NAME} --npz')


RUN_FULLGRID_TRAINING=False. Commands to run:
nnUNetv2_train 701 3d_fullres 0 -p nnUNetPlans_fullgrid192_bs1 --npz


## Experiment 2: increase lesion-positive pressure

The current mixed model is much weaker on ATLAS than ARC, and small lesions tend to be the hardest cases. nnU-Net has built-in trainer variants we can use without writing custom code.

First try `nnUNetTrainer_probabilisticOversampling_033`, which samples foreground-containing patches more aggressively. Use it with the full-grid patch plan if batch size 1 fits, otherwise use `nnUNetPlans_24GB`.


In [12]:
OVERSAMPLING_TRAINER = 'nnUNetTrainer_probabilisticOversampling_033'
OVERSAMPLING_PLANS = BASE_PLANS_NAME

if RUN_OVERSAMPLING_TRAINING:
    for fold in TRAIN_FOLDS:
        run_nnunet_train(layout, fold, OVERSAMPLING_PLANS, trainer=OVERSAMPLING_TRAINER, save_npz=True)
else:
    print('RUN_OVERSAMPLING_TRAINING=False. Commands to run:')
    for fold in TRAIN_FOLDS:
        print(f'nnUNetv2_train {layout.dataset_id} {CONFIGURATION} {fold} -p {OVERSAMPLING_PLANS} -tr {OVERSAMPLING_TRAINER} --npz')


RUN_OVERSAMPLING_TRAINING=False. Commands to run:
nnUNetv2_train 701 3d_fullres 0 -p nnUNetPlans_24GB -tr nnUNetTrainer_probabilisticOversampling_033 --npz


## What the first follow-up experiments showed

Fold 0 comparisons against the original mixed baseline:

- Baseline mixed `nnUNetPlans_24GB`: mean Dice `0.5181`, median `0.6856`.
- Fullgrid `[192, 224, 192]`, batch `1`: mean Dice `0.4754`, median `0.6246`.
- Probabilistic oversampling `033` with `nnUNetPlans_24GB`: mean Dice `0.5123`, median `0.6923`.

Interpretation:

- Bigger patches were actively worse. They mostly added padded/background context because the planned median crop is only `[156, 186, 148]`, and batch size dropped to `1`, reducing sampling stability.
- Oversampling was not a clear win. It slightly helped some `100-999` and `1000-9999` voxel lesions, but hurt enough large ATLAS lesions that mean Dice dropped below baseline.
- The repeated pattern is domain-specific: mixed training remains much better on ARC than ATLAS. The next clean test is an ATLAS-only specialist using the original `nnUNetPlans_24GB` plan and standard trainer.


## Experiment 3: ATLAS-only specialist

The leaderboard comparison is ATLAS lesion segmentation. The mixed v4 training set includes ARC and approximate masks, and the held-out summary shows ARC is much easier for the current model. This experiment trains a second nnU-Net only on rows whose slug contains `ATLAS-Images-f0d7431e`.

Use this model either by itself for ATLAS-only evaluation, or route predictions by source: ATLAS cases use the ATLAS specialist; ARC cases use the mixed model.


In [13]:
ATLAS_ONLY_ROOT = PROJECT_ROOT / 'data/splits/90_10_random/train/nnunet_view_atlas_only'
atlas_layout = pipe.NnUNetLayout(
    project_root=PROJECT_ROOT,
    nnunet_root=ATLAS_ONLY_ROOT,
    dataset_id=702,
    dataset_name='ARC_ATLAS_TrainV4Native_ATLASOnly',
)
ATLAS_SOURCE_FILTER = 'ATLAS-Images-f0d7431e'
ATLAS_PLANS_NAME = 'nnUNetPlans_24GB'
ATLAS_TRAINING_PLANS = ATLAS_PLANS_NAME
ATLAS_TRAINER = None

if RUN_ATLAS_ONLY_PREP:
    summary = pipe.prepare_nnunet_dataset(
        TRAIN_MANIFEST,
        atlas_layout,
        source_filter=ATLAS_SOURCE_FILTER,
        overwrite=True,
        n_splits=5,
    )
    print(json.dumps(summary, indent=2))
else:
    print('RUN_ATLAS_ONLY_PREP=False. ATLAS-only root:', ATLAS_ONLY_ROOT)

if RUN_ATLAS_ONLY_PLAN:
    pipe.plan_and_preprocess(
        atlas_layout,
        verify=True,
        configurations=(CONFIGURATION,),
        gpu_memory_target=24,
        overwrite_plans_name=ATLAS_PLANS_NAME,
    )
else:
    print('RUN_ATLAS_ONLY_PLAN=False.')


{
  "source_manifest": "/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/manifest.csv",
  "dataset_dir": "/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/nnunet_view_atlas_only/nnUNet_raw/Dataset702_ARC_ATLAS_TrainV4Native_ATLASOnly",
  "preprocessed_dir": "/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/nnunet_view_atlas_only/nnUNet_preprocessed/Dataset702_ARC_ATLAS_TrainV4Native_ATLASOnly",
  "results_dir": "/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/nnunet_view_atlas_only/nnUNet_results",
  "dataset_id": 702,
  "dataset_name": "Dataset702_ARC_ATLAS_TrainV4Native_ATLASOnly",
  "num_cases": 582,
  "copy_mode": "symlink",
  "folds": [
    {
      "fold": 0,
      "train": 465,
      "val": 117
    },
    {
      "fold": 1,
      "train": 465,
      "val": 117
    },
    {
      "fo

Extracting dataset fingerprint: 100%|██████████| 582/582 [00:26<00:00, 21.63it/s]


Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [156. 185. 148.], 3d_lowres: [156, 185, 148]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_24GB_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 318, 'patch_size': (np.int64(192), np.int64(160)), 'median_image_size_in_voxels': array([185., 148.]), 'spacing': array([1., 1.]), 'normalization_schemes': ['ZScoreNormalization'], 'use_mask_for_norm': [True], 'resampling_fn_data': 'resample_data_or_seg_to_shape', 'resampling_fn_seg': 'resample_data_or_seg_to_shape', 'resampling_fn_data_kwargs': {'is_seg': False, 'order': 3, 'order_z': 0, 'force_separate_z': None}, 'resam

Preprocessing cases: 100%|██████████| 582/582 [04:26<00:00,  2.18it/s]


In [14]:
# ATLAS-only specialist: intentionally standard plan/trainer first.
# This isolates the domain-specialist effect from the failed fullgrid and mixed oversampling changes.
if RUN_ATLAS_ONLY_TRAINING:
    for fold in TRAIN_FOLDS:
        run_nnunet_train(atlas_layout, fold, ATLAS_TRAINING_PLANS, trainer=ATLAS_TRAINER, save_npz=True)
else:
    print('RUN_ATLAS_ONLY_TRAINING=False. Commands to run after prep/plan:')
    trainer_flag = f' -tr {ATLAS_TRAINER}' if ATLAS_TRAINER else ''
    for fold in TRAIN_FOLDS:
        print(f'nnUNetv2_train {atlas_layout.dataset_id} {CONFIGURATION} {fold} -p {ATLAS_TRAINING_PLANS}{trainer_flag} --npz')


/home/rbielski/miniconda3/envs/tf_310/bin/nnUNetv2_train 702 3d_fullres 0 -p nnUNetPlans_24GB --npz
Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-05-14 11:29:56.417526: Using torch.compile...
2026-05-14 11:29:56.996154: do_dummy_2d_data_aug: False
2026-05-14 11:29:56.997218: Using splits from existing split file: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/nnunet_view_atlas_only/nnUNet_preprocessed/Dataset702_ARC_ATLAS_TrainV4Native_ATLASOnly/splits_final.json
2026-05-14 11:29:56.997489: The split file contains 5 splits.
2026-05-14 11:29:56.997523: Desired

## Experiment 4: evaluation routing

After `ARC_ATLAS_Test_v4.ipynb` has produced mixed-model predictions on v3 `test_hires`, and after the ATLAS-only model has produced ATLAS predictions, make a routed result:

- `ATLAS` cases: ATLAS-only specialist prediction.
- `ARC` cases: current mixed nnU-Net prediction.

This directly tests whether the mixed-domain training is suppressing ATLAS performance.


In [15]:
# Optional routing utility. Run only after both prediction directories exist.
MIXED_V3_PRED_DIR = PROJECT_ROOT / 'runs/nnunet_v3_test_hires_predictions_native'
ATLAS_V3_PRED_DIR = PROJECT_ROOT / 'runs/nnunet_v3_test_hires_predictions_atlas_specialist'
ROUTED_V3_PRED_DIR = PROJECT_ROOT / 'runs/nnunet_v3_test_hires_predictions_routed'
V3_INPUT_MAPPING = PROJECT_ROOT / 'runs/nnunet_v3_test_hires_input_native/case_mapping.csv'
RUN_ROUTED_OUTPUT = False

if RUN_ROUTED_OUTPUT:
    mapping = pd.read_csv(V3_INPUT_MAPPING)
    ROUTED_V3_PRED_DIR.mkdir(parents=True, exist_ok=True)
    for _, row in mapping.iterrows():
        case_id = row['case_id']
        source = str(row.get('source_slug', '')).upper()
        src_dir = ATLAS_V3_PRED_DIR if 'ATLAS' in source else MIXED_V3_PRED_DIR
        src = src_dir / f'{case_id}.nii.gz'
        dst = ROUTED_V3_PRED_DIR / f'{case_id}.nii.gz'
        if not src.exists():
            raise FileNotFoundError(src)
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        dst.symlink_to(src)
    print('Wrote routed prediction links:', ROUTED_V3_PRED_DIR)
else:
    print('RUN_ROUTED_OUTPUT=False.')


RUN_ROUTED_OUTPUT=False.


## What to change if full-grid batch 1 OOMs

Try these in order:

1. Keep `[192, 224, 192]`, batch size `1`, but set `torch.set_float32_matmul_precision('medium')` is irrelevant here; nnU-Net already uses mixed precision. If it OOMs, the patch is simply too large for this network/deep supervision setup.
2. Reduce to nearest conservative full-ish patch, for example `[176, 208, 176]` or current `[160, 192, 160]` plus oversampling.
3. Do not switch datasets. Keep the same manifest and same nnU-Net raw/preprocessed root.
4. Avoid the old probability postprocessing until its geometry/probability interpretation is fixed.

For raw shape coverage, `[192, 224, 192]` is the v3 grid but is slightly smaller than the raw `193 x 229 x 193` files. The current nnU-Net inference still writes full-size masks. If you want a patch that numerically covers raw dimensions with divisibility headroom, `[208, 240, 208]` is the next obvious candidate, but it is larger and should only be tried after `[192, 224, 192]` batch 1 works.
